In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("fraud-streaming") \
    .config("spark.jars.packages",
            "io.delta:delta-spark_2.12:3.1.0,"
            "org.apache.hadoop:hadoop-aws:3.3.4,"
            "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
            "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

print("✅ Spark session ready")

✅ Spark session ready


In [2]:
from pyspark.sql.functions import from_json, col, window, sum as _sum, count, avg
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, TimestampType

schema = StructType([
    StructField("transaction_id", StringType()),
    StructField("type", StringType()),
    StructField("amount", DoubleType()),
    StructField("nameOrig", StringType()),
    StructField("nameDest", StringType()),
    StructField("oldbalanceOrg", DoubleType()),
    StructField("newbalanceOrig", DoubleType()),
    StructField("isFraud", IntegerType()),
    StructField("timestamp", StringType()),
    StructField("source", StringType()),
])

raw = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("subscribe", "transactions") \
    .option("startingOffsets", "latest") \
    .load()

parsed = raw.select(
    from_json(col("value").cast("string"), schema).alias("data"),
    col("timestamp").alias("kafka_timestamp")
).select("data.*", "kafka_timestamp")

print("✅ Streaming reader defined")

✅ Streaming reader defined


In [3]:
checkpoint_path = "s3a://silver/checkpoints/streaming/"
output_path = "s3a://silver/streaming/transactions/"

query = parsed.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", checkpoint_path) \
    .option("path", output_path) \
    .trigger(processingTime="10 seconds") \
    .start()

print("✅ Streaming query started — writing to silver Delta table")
print(f"Status: {query.status}")

✅ Streaming query started — writing to silver Delta table
Status: {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}


In [6]:
print(query.status)
print(query.lastProgress)

{'message': 'Getting offsets from KafkaV2[Subscribe[transactions]]', 'isDataAvailable': False, 'isTriggerActive': True}
None


In [8]:
print(query.status)
print(query.lastProgress)

{'message': 'Terminated with exception: org.apache.kafka.common.errors.TimeoutException: Timed out waiting for a node assignment. Call: describeTopics', 'isDataAvailable': False, 'isTriggerActive': False}
None


In [9]:
query.stop()

In [10]:
raw = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:29092") \
    .option("subscribe", "transactions") \
    .option("startingOffsets", "latest") \
    .load()

parsed = raw.select(
    from_json(col("value").cast("string"), schema).alias("data"),
    col("timestamp").alias("kafka_timestamp")
).select("data.*", "kafka_timestamp")

query = parsed.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "s3a://silver/checkpoints/streaming/") \
    .option("path", "s3a://silver/streaming/transactions/") \
    .trigger(processingTime="10 seconds") \
    .start()

print(f"✅ Stream started — status: {query.status}")

✅ Stream started — status: {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}


In [11]:
print(query.status)
print(query.lastProgress)

{'message': 'Getting offsets from KafkaV2[Subscribe[transactions]]', 'isDataAvailable': False, 'isTriggerActive': True}
None


In [12]:
print(query.status)
print(query.lastProgress)

{'message': 'Getting offsets from KafkaV2[Subscribe[transactions]]', 'isDataAvailable': False, 'isTriggerActive': True}
None


In [13]:
query.stop()

In [14]:
raw = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:29092") \
    .option("subscribe", "transactions") \
    .option("startingOffsets", "latest") \
    .load()

parsed = raw.select(
    from_json(col("value").cast("string"), schema).alias("data"),
    col("timestamp").alias("kafka_timestamp")
).select("data.*", "kafka_timestamp")

query = parsed.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "s3a://silver/checkpoints/streaming/") \
    .option("path", "s3a://silver/streaming/transactions/") \
    .trigger(processingTime="10 seconds") \
    .start()

print(f"✅ Stream started — status: {query.status}")

✅ Stream started — status: {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}


In [15]:
print(query.status)
print(query.lastProgress)

{'message': 'Waiting for next trigger', 'isDataAvailable': True, 'isTriggerActive': False}
{'id': '4d48a204-fcaf-4e9f-8ac4-ba265b7db7c3', 'runId': '0e043edf-88b8-459b-832e-8327cdb3d220', 'name': None, 'timestamp': '2026-05-26T17:58:10.000Z', 'batchId': 8, 'numInputRows': 99, 'inputRowsPerSecond': 9.9, 'processedRowsPerSecond': 38.976377952755904, 'durationMs': {'addBatch': 2377, 'commitOffsets': 87, 'getBatch': 0, 'latestOffset': 8, 'queryPlanning': 9, 'triggerExecution': 2540, 'walCommit': 58}, 'stateOperators': [], 'sources': [{'description': 'KafkaV2[Subscribe[transactions]]', 'startOffset': {'transactions': {'2': 1593, '5': 1619, '4': 1617, '7': 1567, '1': 1577, '3': 1501, '6': 1581, '0': 1553}}, 'endOffset': {'transactions': {'2': 1605, '5': 1630, '4': 1629, '7': 1578, '1': 1588, '3': 1517, '6': 1594, '0': 1566}}, 'latestOffset': {'transactions': {'2': 1605, '5': 1630, '4': 1629, '7': 1578, '1': 1588, '3': 1517, '6': 1594, '0': 1566}}, 'numInputRows': 99, 'inputRowsPerSecond': 9.9

In [16]:
df = spark.read.format("delta").load("s3a://silver/streaming/transactions/")
print(f"✅ Rows in silver streaming table: {df.count()}")
df.show(5)

✅ Rows in silver streaming table: 1305
+--------------+--------+-------+--------+--------+-------------+--------------+-------+--------------------+---------+--------------------+
|transaction_id|    type| amount|nameOrig|nameDest|oldbalanceOrg|newbalanceOrig|isFraud|           timestamp|   source|     kafka_timestamp|
+--------------+--------+-------+--------+--------+-------------+--------------+-------+--------------------+---------+--------------------+
|       T344884|CASH_OUT|2738.56|C2813106|C4549325|    399908.32|      511044.4|      0|2026-05-26T17:57:...|synthetic|2026-05-26 17:57:...|
|       T171697|TRANSFER|4541.53|C1345411|C2200294|    719167.12|     975987.81|      0|2026-05-26T17:57:...|synthetic|2026-05-26 17:57:...|
|       T366097| CASH_IN|1042.88|C4505947|C4116070|    204117.43|     853590.05|      0|2026-05-26T17:57:...|synthetic|2026-05-26 17:57:...|
|       T495086|TRANSFER|3800.12|C4529508|C4414494|    845273.45|      270746.4|      0|2026-05-26T17:57:...|synthe

In [17]:
from pyspark.sql.functions import to_timestamp, window, col, sum as _sum, count, avg

# Stop current query first
query.stop()

# Re-read stream with watermark
raw = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:29092") \
    .option("subscribe", "transactions") \
    .option("startingOffsets", "latest") \
    .load()

parsed = raw.select(
    from_json(col("value").cast("string"), schema).alias("data"),
    col("timestamp").alias("kafka_timestamp")
).select("data.*", "kafka_timestamp")

# Add proper timestamp and watermark for late data (10 minute tolerance)
with_watermark = parsed \
    .withColumn("event_time", to_timestamp(col("timestamp"))) \
    .withWatermark("event_time", "10 minutes")

# Sliding window aggregations per card per 1h window
windowed = with_watermark.groupBy(
    window(col("event_time"), "1 hour", "10 minutes"),
    col("nameOrig")
).agg(
    count("transaction_id").alias("tx_count_1h"),
    _sum("amount").alias("total_amount_1h"),
    avg("amount").alias("avg_amount_1h"),
    _sum("isFraud").alias("fraud_count_1h")
)

query_windowed = windowed.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "s3a://silver/checkpoints/windowed/") \
    .option("path", "s3a://silver/streaming/windowed_features/") \
    .trigger(processingTime="10 seconds") \
    .start()

print(f"✅ Windowed stream started: {query_windowed.status}")

✅ Windowed stream started: {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}


In [18]:
df_w = spark.read.format("delta").load("s3a://silver/streaming/windowed_features/")
print(f"Windowed features rows: {df_w.count()}")
df_w.show(5, truncate=False)

Windowed features rows: 0
+------+--------+-----------+---------------+-------------+--------------+
|window|nameOrig|tx_count_1h|total_amount_1h|avg_amount_1h|fraud_count_1h|
+------+--------+-----------+---------------+-------------+--------------+
+------+--------+-----------+---------------+-------------+--------------+



In [19]:
df_w = spark.read.format("delta").load("s3a://silver/streaming/windowed_features/")
print(f"Windowed features rows: {df_w.count()}")
df_w.show(5, truncate=False)

Windowed features rows: 0
+------+--------+-----------+---------------+-------------+--------------+
|window|nameOrig|tx_count_1h|total_amount_1h|avg_amount_1h|fraud_count_1h|
+------+--------+-----------+---------------+-------------+--------------+
+------+--------+-----------+---------------+-------------+--------------+



In [20]:
import boto3
s3 = boto3.client("s3", endpoint_url="http://minio:9000",
                  aws_access_key_id="minioadmin", aws_secret_access_key="minioadmin")
paginator = s3.get_paginator("list_objects_v2")
for page in paginator.paginate(Bucket="silver", Prefix="checkpoints/windowed/"):
    for obj in page.get("Contents", []):
        s3.delete_object(Bucket="silver", Key=obj["Key"])
print("✅ Checkpoint cleared")

✅ Checkpoint cleared


In [21]:
query_windowed.stop()

In [24]:
query_windowed = windowed.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "s3a://silver/checkpoints/windowed/") \
    .option("path", "s3a://silver/streaming/windowed_features/") \
    .trigger(processingTime="10 seconds") \
    .start()

In [25]:
df_check = spark.read.format("delta") \
    .load("s3a://silver/streaming/windowed_features/")

df_check.show(truncate=False)

+------+--------+-----------+---------------+-------------+--------------+
|window|nameOrig|tx_count_1h|total_amount_1h|avg_amount_1h|fraud_count_1h|
+------+--------+-----------+---------------+-------------+--------------+
+------+--------+-----------+---------------+-------------+--------------+



In [26]:
import boto3
s3 = boto3.client("s3", endpoint_url="http://minio:9000",
                  aws_access_key_id="minioadmin", aws_secret_access_key="minioadmin")
paginator = s3.get_paginator("list_objects_v2")
for page in paginator.paginate(Bucket="silver", Prefix="checkpoints/windowed/"):
    for obj in page.get("Contents", []):
        s3.delete_object(Bucket="silver", Key=obj["Key"])
print("✅ Checkpoint cleared")

✅ Checkpoint cleared


In [27]:
query_windowed.stop()

query_windowed = windowed.writeStream \
    .format("delta") \
    .outputMode("update") \
    .option("checkpointLocation", "s3a://silver/checkpoints/windowed/") \
    .option("path", "s3a://silver/streaming/windowed_features/") \
    .trigger(processingTime="10 seconds") \
    .start()
print(query_windowed.status)

AnalysisException: [DELTA_UNSUPPORTED_OUTPUT_MODE] Data source org.apache.spark.sql.delta.sources.DeltaDataSource does not support Update output mode

In [28]:
query_windowed.stop()

# Clear checkpoint
import boto3
s3 = boto3.client("s3", endpoint_url="http://minio:9000",
                  aws_access_key_id="minioadmin", aws_secret_access_key="minioadmin")
paginator = s3.get_paginator("list_objects_v2")
for page in paginator.paginate(Bucket="silver", Prefix="checkpoints/windowed/"):
    for obj in page.get("Contents", []):
        s3.delete_object(Bucket="silver", Key=obj["Key"])
print("✅ Checkpoint cleared")

# Redefine with SHORT window for testing
windowed = with_watermark.groupBy(
    window(col("event_time"), "1 minute", "30 seconds"),
    col("nameOrig")
).agg(
    count("transaction_id").alias("tx_count_1h"),
    _sum("amount").alias("total_amount_1h"),
    avg("amount").alias("avg_amount_1h"),
    _sum("isFraud").alias("fraud_count_1h")
)

query_windowed = windowed.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "s3a://silver/checkpoints/windowed/") \
    .option("path", "s3a://silver/streaming/windowed_features/") \
    .trigger(processingTime="10 seconds") \
    .start()

print("✅ Windowed stream started:", query_windowed.status)

✅ Checkpoint cleared
✅ Windowed stream started: {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}


In [29]:
df_w = spark.read.format("delta").load("s3a://silver/streaming/windowed_features/")
print(f"Rows: {df_w.count()}")
df_w.show(5, truncate=False)

Rows: 0
+------+--------+-----------+---------------+-------------+--------------+
|window|nameOrig|tx_count_1h|total_amount_1h|avg_amount_1h|fraud_count_1h|
+------+--------+-----------+---------------+-------------+--------------+
+------+--------+-----------+---------------+-------------+--------------+



In [30]:
print(query_windowed.status)
print(query_windowed.lastProgress)

{'message': 'Processing new data', 'isDataAvailable': True, 'isTriggerActive': True}
{'id': 'b18d3001-ca03-47ca-9e14-4f38ff37d579', 'runId': '548045fe-9704-4541-b466-0b33b43e817f', 'name': None, 'timestamp': '2026-05-26T22:29:50.000Z', 'batchId': 25, 'numInputRows': 100, 'inputRowsPerSecond': 10.0, 'processedRowsPerSecond': 10.753844499408538, 'durationMs': {'addBatch': 9215, 'commitOffsets': 43, 'getBatch': 0, 'latestOffset': 1, 'queryPlanning': 8, 'triggerExecution': 9299, 'walCommit': 31}, 'eventTime': {'avg': '2026-05-26T22:29:44.990Z', 'max': '2026-05-26T22:29:49.963Z', 'min': '2026-05-26T22:29:40.016Z', 'watermark': '2026-05-26T22:19:39.915Z'}, 'stateOperators': [{'operatorName': 'stateStoreSave', 'numRowsTotal': 4432, 'numRowsUpdated': 200, 'allUpdatesTimeMs': 2357, 'numRowsRemoved': 0, 'allRemovalsTimeMs': 686, 'commitTimeMs': 12670, 'memoryUsedBytes': 1465344, 'numRowsDroppedByWatermark': 0, 'numShufflePartitions': 200, 'numStateStoreInstances': 200, 'customMetrics': {'loadedM

In [31]:
df_w = spark.read.format("delta").load("s3a://silver/streaming/windowed_features/")
print(f"Rows: {df_w.count()}")
df_w.show(5, truncate=False)

Rows: 0
+------+--------+-----------+---------------+-------------+--------------+
|window|nameOrig|tx_count_1h|total_amount_1h|avg_amount_1h|fraud_count_1h|
+------+--------+-----------+---------------+-------------+--------------+
+------+--------+-----------+---------------+-------------+--------------+



In [32]:
query_windowed.stop()

query_windowed_mem = windowed.writeStream \
    .format("memory") \
    .queryName("windowed_check") \
    .outputMode("complete") \
    .trigger(processingTime="10 seconds") \
    .start()

print("✅ Memory stream started:", query_windowed_mem.status)

✅ Memory stream started: {'message': 'Getting offsets from KafkaV2[Subscribe[transactions]]', 'isDataAvailable': False, 'isTriggerActive': True}


In [33]:
spark.sql("SELECT * FROM windowed_check ORDER BY window DESC LIMIT 10").show(truncate=False)

+------------------------------------------+--------+-----------+---------------+-------------+--------------+
|window                                    |nameOrig|tx_count_1h|total_amount_1h|avg_amount_1h|fraud_count_1h|
+------------------------------------------+--------+-----------+---------------+-------------+--------------+
|{2026-05-26 22:32:30, 2026-05-26 22:33:30}|C4136743|1          |411.82         |411.82       |0             |
|{2026-05-26 22:32:30, 2026-05-26 22:33:30}|C1477357|1          |1994.14        |1994.14      |0             |
|{2026-05-26 22:32:30, 2026-05-26 22:33:30}|C3973224|1          |593.57         |593.57       |0             |
|{2026-05-26 22:32:30, 2026-05-26 22:33:30}|C4322569|1          |4315.77        |4315.77      |0             |
|{2026-05-26 22:32:30, 2026-05-26 22:33:30}|C9924003|1          |2975.27        |2975.27      |0             |
|{2026-05-26 22:32:30, 2026-05-26 22:33:30}|C2890397|1          |3948.87        |3948.87      |0             |
|

In [34]:
query_windowed_mem.stop()
query_windowed.stop()

# Clear checkpoint
import boto3
s3 = boto3.client("s3", endpoint_url="http://minio:9000",
                  aws_access_key_id="minioadmin", aws_secret_access_key="minioadmin")
paginator = s3.get_paginator("list_objects_v2")
for page in paginator.paginate(Bucket="silver", Prefix="checkpoints/windowed/"):
    for obj in page.get("Contents", []):
        s3.delete_object(Bucket="silver", Key=obj["Key"])
print("✅ Checkpoint cleared")

# Redefine with short watermark
with_watermark = parsed \
    .withColumn("event_time", to_timestamp(col("timestamp"))) \
    .withWatermark("event_time", "30 seconds")  # ← was 10 minutes

windowed = with_watermark.groupBy(
    window(col("event_time"), "1 minute", "30 seconds"),
    col("nameOrig")
).agg(
    count("transaction_id").alias("tx_count_1h"),
    _sum("amount").alias("total_amount_1h"),
    avg("amount").alias("avg_amount_1h"),
    _sum("isFraud").alias("fraud_count_1h")
)

query_windowed = windowed.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "s3a://silver/checkpoints/windowed/") \
    .option("path", "s3a://silver/streaming/windowed_features/") \
    .trigger(processingTime="10 seconds") \
    .start()

print("✅ Stream started:", query_windowed.status)

✅ Checkpoint cleared
✅ Stream started: {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}


In [35]:
df_w = spark.read.format("delta").load("s3a://silver/streaming/windowed_features/")
print(f"Rows: {df_w.count()}")
df_w.show(5, truncate=False)

Rows: 2214
+------------------------------------------+--------+-----------+---------------+-------------+--------------+
|window                                    |nameOrig|tx_count_1h|total_amount_1h|avg_amount_1h|fraud_count_1h|
+------------------------------------------+--------+-----------+---------------+-------------+--------------+
|{2026-05-26 22:35:00, 2026-05-26 22:36:00}|C6319692|1          |1789.4         |1789.4       |0             |
|{2026-05-26 22:35:00, 2026-05-26 22:36:00}|C6120693|1          |4207.32        |4207.32      |0             |
|{2026-05-26 22:35:00, 2026-05-26 22:36:00}|C2528630|1          |2667.79        |2667.79      |0             |
|{2026-05-26 22:35:00, 2026-05-26 22:36:00}|C2176656|1          |3601.0         |3601.0       |0             |
|{2026-05-26 22:35:30, 2026-05-26 22:36:30}|C1335933|1          |182.58         |182.58       |0             |
+------------------------------------------+--------+-----------+---------------+-------------+------

In [38]:
import subprocess
subprocess.run(["pip", "install", "confluent-kafka"], capture_output=True)

from confluent_kafka import Producer
import json

def send_fraud_alerts(batch_df, batch_id):
    fraud = batch_df.filter(col("isFraud") == 1)
    count = fraud.count()
    if count > 0:
        producer = Producer({"bootstrap.servers": "kafka:29092"})
        for row in fraud.collect():
            producer.produce(
                "fraud-alerts",
                value=json.dumps({
                    "transaction_id": row["transaction_id"],
                    "amount": row["amount"],
                    "nameOrig": row["nameOrig"],
                    "nameDest": row["nameDest"],
                    "type": row["type"],
                    "timestamp": str(row["timestamp"])
                }).encode("utf-8")
            )
        producer.flush()
        print(f"🚨 Batch {batch_id}: sent {count} fraud alerts")

query_alerts = parsed.writeStream \
    .foreachBatch(send_fraud_alerts) \
    .option("checkpointLocation", "s3a://silver/checkpoints/fraud_alerts/") \
    .trigger(processingTime="10 seconds") \
    .start()

print("✅ Fraud alerts stream started:", query_alerts.status)

✅ Fraud alerts stream started: {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}


🚨 Batch 2: sent 1 fraud alerts
🚨 Batch 3: sent 2 fraud alerts
🚨 Batch 4: sent 1 fraud alerts
🚨 Batch 5: sent 3 fraud alerts
🚨 Batch 6: sent 1 fraud alerts
🚨 Batch 9: sent 1 fraud alerts
🚨 Batch 12: sent 1 fraud alerts
🚨 Batch 13: sent 1 fraud alerts


In [39]:
from confluent_kafka import Consumer
import json

consumer = Consumer({
    "bootstrap.servers": "kafka:29092",
    "group.id": "fraud-alert-verifier",
    "auto.offset.reset": "earliest"
})
consumer.subscribe(["fraud-alerts"])

print("Reading fraud alerts from Kafka...")
alerts = []
for _ in range(30):
    msg = consumer.poll(timeout=1.0)
    if msg and not msg.error():
        alerts.append(json.loads(msg.value()))

consumer.close()
print(f"✅ Found {len(alerts)} fraud alerts in topic")
for a in alerts[:3]:
    print(a)

Reading fraud alerts from Kafka...
🚨 Batch 14: sent 1 fraud alerts
🚨 Batch 15: sent 2 fraud alerts
✅ Found 14 fraud alerts in topic
{'transaction_id': 'T487629', 'amount': 44292.53, 'nameOrig': 'C6391503', 'nameDest': 'C9946082', 'type': 'CASH_IN', 'timestamp': '2026-05-26T22:42:17.413459'}
{'transaction_id': 'T299265', 'amount': 10315.97, 'nameOrig': 'C9804109', 'nameDest': 'C1692721', 'type': 'CASH_OUT', 'timestamp': '2026-05-26T22:42:27.657681'}
{'transaction_id': 'T339633', 'amount': 32292.96, 'nameOrig': 'C6937291', 'nameDest': 'C8908206', 'type': 'DEBIT', 'timestamp': '2026-05-26T22:41:52.876540'}


🚨 Batch 16: sent 1 fraud alerts
🚨 Batch 17: sent 1 fraud alerts
🚨 Batch 18: sent 1 fraud alerts
🚨 Batch 20: sent 2 fraud alerts
🚨 Batch 22: sent 2 fraud alerts
🚨 Batch 27: sent 1 fraud alerts
🚨 Batch 28: sent 1 fraud alerts
🚨 Batch 35: sent 2 fraud alerts
🚨 Batch 36: sent 2 fraud alerts
🚨 Batch 38: sent 2 fraud alerts


In [40]:
from confluent_kafka.admin import AdminClient
from confluent_kafka import Consumer, TopicPartition

def get_kafka_lag(topic, group_id):
    consumer = Consumer({
        "bootstrap.servers": "kafka:29092",
        "group.id": group_id,
        "auto.offset.reset": "earliest"
    })
    admin = AdminClient({"bootstrap.servers": "kafka:29092"})
    
    metadata = admin.list_topics(topic=topic)
    partitions = [TopicPartition(topic, p) for p in metadata.topics[topic].partitions]
    
    committed = consumer.committed(partitions)
    end_offsets = consumer.get_watermark_offsets
    
    total_lag = 0
    for tp in committed:
        low, high = consumer.get_watermark_offsets(tp)
        committed_offset = tp.offset if tp.offset >= 0 else low
        lag = high - committed_offset
        total_lag += lag
        print(f"  Partition {tp.partition}: committed={committed_offset}, latest={high}, lag={lag}")
    
    consumer.close()
    print(f"Total lag for '{topic}': {total_lag}")

print("=== transactions topic ===")
get_kafka_lag("transactions", "spark-streaming-group")

print("\n=== fraud-alerts topic ===")
get_kafka_lag("fraud-alerts", "fraud-alert-verifier")

=== transactions topic ===
  Partition 0: committed=0, latest=6546, lag=6546
  Partition 1: committed=0, latest=6462, lag=6462
  Partition 2: committed=0, latest=6427, lag=6427
  Partition 3: committed=0, latest=6411, lag=6411
  Partition 4: committed=0, latest=6478, lag=6478
  Partition 5: committed=0, latest=6480, lag=6480
  Partition 6: committed=0, latest=6434, lag=6434
  Partition 7: committed=0, latest=6523, lag=6523
Total lag for 'transactions': 51761

=== fraud-alerts topic ===
  Partition 0: committed=2, latest=4, lag=2
  Partition 1: committed=3, latest=4, lag=1
  Partition 2: committed=1, latest=2, lag=1
  Partition 3: committed=3, latest=4, lag=1
  Partition 4: committed=2, latest=3, lag=1
  Partition 5: committed=0, latest=3, lag=3
  Partition 6: committed=1, latest=3, lag=2
  Partition 7: committed=2, latest=6, lag=4
Total lag for 'fraud-alerts': 15


🚨 Batch 40: sent 1 fraud alerts
🚨 Batch 42: sent 1 fraud alerts
🚨 Batch 43: sent 1 fraud alerts
🚨 Batch 46: sent 1 fraud alerts
🚨 Batch 48: sent 3 fraud alerts
🚨 Batch 49: sent 1 fraud alerts
🚨 Batch 50: sent 2 fraud alerts
🚨 Batch 51: sent 3 fraud alerts
🚨 Batch 55: sent 1 fraud alerts
🚨 Batch 56: sent 3 fraud alerts
🚨 Batch 57: sent 1 fraud alerts
🚨 Batch 61: sent 1 fraud alerts
🚨 Batch 62: sent 1 fraud alerts
🚨 Batch 63: sent 1 fraud alerts
🚨 Batch 64: sent 3 fraud alerts
🚨 Batch 65: sent 1 fraud alerts
🚨 Batch 68: sent 1 fraud alerts
🚨 Batch 70: sent 2 fraud alerts
🚨 Batch 71: sent 1 fraud alerts
🚨 Batch 72: sent 1 fraud alerts
🚨 Batch 74: sent 2 fraud alerts
🚨 Batch 77: sent 2 fraud alerts
🚨 Batch 78: sent 2 fraud alerts
🚨 Batch 81: sent 1 fraud alerts
🚨 Batch 82: sent 1 fraud alerts
🚨 Batch 83: sent 2 fraud alerts
🚨 Batch 85: sent 3 fraud alerts
🚨 Batch 86: sent 2 fraud alerts
🚨 Batch 87: sent 2 fraud alerts
🚨 Batch 91: sent 1 fraud alerts
🚨 Batch 92: sent 3 fraud alerts
🚨 Batch 